In [8]:
from data import *

In [22]:
import copy
import math
import numpy as np
from data import Data

# -------------------------------------------------------
# 1) triangulation + compute_pfd 거리 기반 1-median coreset
# -------------------------------------------------------

def build_triangulation_coreset_practical(
    data,
    eps: float = 0.1,
    alpha: float = 8.0,
    alpha_min: float = 16.0,
    center_candidates: int = 5,
):
    """
    data.triangulations 위에서 compute_pfd를 거리로 사용하는
    1-median coreset 생성기.

    반환:
      S_idx      : 대표 triangulation 인덱스들 (np.ndarray[int])
      S_weights  : 각 대표가 대표하는 개수 (np.ndarray[int])
    """

    n = len(data.triangulations)
    if n == 0:
        return np.array([], dtype=int), np.array([], dtype=int)

    # --- compute_pfd(i, j) 캐시 ---
    pfd_cache = {}

    def get_pfd(i, j):
        if i == j:
            return 0.0
        a, b = (i, j) if i < j else (j, i)
        key = (a, b)
        if key not in pfd_cache:
            steps, _, _, _ = data.compute_pfd(a, b)
            pfd_cache[key] = float(steps)
        return pfd_cache[key]

    # --- 1. 대략적인 center 후보 하나 잡기 (discrete 1-median 근사) ---
    m = n
    cand_num = min(m, center_candidates)
    best_center_idx = 0
    best_total_dist = float("inf")

    for c in range(cand_num):
        total = 0.0
        for j in range(m):
            total += get_pfd(c, j)
        if total < best_total_dist:
            best_total_dist = total
            best_center_idx = c

    center_idx = best_center_idx

    # --- 2. center_idx 기준 거리 배열 d(i) = pfd(center_idx, i) ---
    nearest_dist = np.zeros(m, dtype=float)
    for i in range(m):
        nearest_dist[i] = get_pfd(center_idx, i)

    nu_A = float(nearest_dist.sum())
    R = max(nu_A / max(1, m), 1e-12)
    Dmax = float(nearest_dist.max())

    # 모든 triangulation이 center와 너무 가깝다면 -> 하나로 요약
    if Dmax < R:
        return np.array([center_idx], dtype=int), np.array([m], dtype=int)

    # --- 3. 거리 스케일(ring) + 1D grid로 버킷팅 ---
    from math import ceil, log2
    M = max(1, ceil(log2((Dmax + R) / R)))

    buckets = {}
    tiny = 1e-12

    for i in range(m):
        dist = nearest_dist[i]
        inf_norm = dist  # 1D라 그냥 dist

        # level j 결정 (R, 2R, 4R, ...)
        if inf_norm < R:
            j = 0
        else:
            j = int(math.ceil(math.log2(max(inf_norm, tiny) / R)))
        j = min(j, M)

        # level j에서의 cell 크기 rj
        rj = (eps * R * (2 ** j)) / alpha
        rj = max(rj, (eps * R) / alpha_min)

        # 1D grid cell index
        cell_1d = int(math.floor(inf_norm / rj))
        key = (j, cell_1d)

        # 같은 버킷에 들어오는 triangulation들을 하나로 묶음
        if key not in buckets:
            buckets[key] = [i, 1]  # 대표 index, count
        else:
            buckets[key][1] += 1

    if buckets:
        reps = []
        wts = []
        for rep_idx, cnt in buckets.values():
            reps.append(rep_idx)
            wts.append(cnt)
        S_idx = np.array(reps, dtype=int)
        S_weights = np.array(wts, dtype=int)
    else:
        # 이론상 거의 안 나오는 경우지만, 방어적으로
        S_idx = np.array([center_idx], dtype=int)
        S_weights = np.array([m], dtype=int)

    return S_idx, S_weights


In [23]:
# -------------------------------------------------------
# 2) 코어셋 triangulation 들만 가지고 find_center 실행
# -------------------------------------------------------

def make_coreset_data(data_full: Data, S_idx):
    """
    data_full과 같은 pts / geometry 정보를 공유하지만,
    triangulations 리스트만 코어셋으로 교체한 Data 객체를 만든다.
    (weights는 일단 find_center에서는 쓰지 않고, 평가할 때만 사용)
    """
    coreset_data = copy.copy(data_full)  # 얕은 복사: pts, etc. 공유

    # 코어셋 triangulation만 복사해서 넣기
    new_tris = [copy.deepcopy(data_full.triangulations[i]) for i in S_idx]
    coreset_data.triangulations = new_tris

    # center / flip / dist 초기화
    coreset_data.center = coreset_data.triangulations[0]
    coreset_data.flip = [[] for _ in coreset_data.triangulations]
    coreset_data.dist = float("inf")

    return coreset_data


In [25]:
# -------------------------------------------------------
# 3) full vs coreset에서 find_center 결과 비교
# -------------------------------------------------------

def compare_center_with_coreset(
    instance_path: str = "random_instance_4_40_2.json",
    eps: float = 0.4,
    alpha: float = 8.0,
    alpha_min: float = 16.0,
    center_candidates: int = 5,
):
    # (A) 원본 인스턴스 로딩
    data_full = Data(instance_path)
    n_full = len(data_full.triangulations)
    print(f"# full triangulations: {n_full}")

    # (B) full 데이터에서 continuous center (find_center)
    print("\n[1] full 데이터에서 center 찾는 중...")
    center_full, dist_full_on_full = data_full.find_center()
    # 안전하게 한 번 더 평가
    dist_full_on_full_check, _ = data_full.compute_center_dist(center_full)
    print(f" full center_dist (full 데이터 기준): {dist_full_on_full_check}")

    # (C) 1-median coreset 생성 (compute_pfd 기반)
    print("\n[2] 1-median coreset 생성 중...")
    S_idx, S_weights = build_triangulation_coreset_practical(
        data_full,
        eps=eps,
        alpha=alpha,
        alpha_min=alpha_min,
        center_candidates=center_candidates,
    )
    print(f" coreset reps: {len(S_idx)} / weights 합: {S_weights.sum()}")

    # (D) 코어셋만 가지고 continuous center (find_center)
    print("\n[3] coreset 데이터에서 center 찾는 중...")
    data_core = make_coreset_data(data_full, S_idx)
    center_core, dist_core_on_core = data_core.find_center()
    print(f" center_dist (coreset 데이터 기준): {dist_core_on_core}")

    # (E) coreset center를 full 데이터에서 평가
    print("\n[4] 두 center를 full 데이터에서 평가...")
    dist_core_on_full, _ = data_full.compute_center_dist(center_core)

    # 결과 요약
    print("\n===== 결과 요약 =====")
    print(f"full center_dist (full 기준):      {dist_full_on_full_check}")
    print(f"coreset center_dist (full 기준):   {dist_core_on_full}")
    abs_diff = dist_core_on_full - dist_full_on_full_check
    ratio = dist_core_on_full / dist_full_on_full_check if dist_full_on_full_check > 0 else float("nan")
    print(f"절대 차이: {abs_diff}")
    print(f"비율(coreset/full): {ratio:.6f}")
    print(f"상대 오차(rel_error): {ratio - 1.0:.6f}")


if __name__ == "__main__":
    # 인스턴스 경로는 실제 위치에 맞게 수정해줘
    compare_center_with_coreset(
        instance_path="./data/benchmark_instances/random_instance_132_80_20.json",
        eps=0.4,
        alpha=8.0,
        alpha_min=16.0,
        center_candidates=5,
    )


--------------------ReadData--------------------
num of pts: 80
num of triangulations: 20
Maximum Parallel flip distance: 9
Initial Center: 9 (total dist: 136)
# full triangulations: 20

[1] full 데이터에서 center 찾는 중...
[1 step] Triangulation 9 flipped, 50 edges
[2 step] Triangulation 15 flipped, 51 edges
[3 step] Triangulation 19 flipped, 52 edges
[4 step] Triangulation 3 flipped, 48 edges
[5 step] Triangulation 0 flipped, 55 edges
[6 step] Triangulation 4 flipped, 50 edges
[7 step] Triangulation 5 flipped, 51 edges
[8 step] Triangulation 14 flipped, 48 edges
[9 step] Triangulation 18 flipped, 54 edges
[10 step] Triangulation 12 flipped, 53 edges
[11 step] Triangulation 6 flipped, 51 edges
[12 step] Triangulation 16 flipped, 51 edges
[13 step] Triangulation 11 flipped, 52 edges
[14 step] Triangulation 17 flipped, 48 edges
[15 step] Triangulation 8 flipped, 51 edges
[16 step] Triangulation 10 flipped, 50 edges
[17 step] Triangulation 7 flipped, 53 edges
[18 step] Triangulation 18 flipped,

In [ ]:
import copy
import math
import numpy as np
from data import Data

# -------------------------------------------------------
# 1. compute_pfd로부터 거리 행렬 D(i,j) 만들기
# -------------------------------------------------------

def build_pfd_distance_matrix(data: Data) -> np.ndarray:
    """
    data.triangulations 에 대해
    D[i, j] = compute_pfd(i, j)의 step 수 (평행 flip distance)
    를 담은 대칭 거리 행렬을 만든다.
    """
    m = len(data.triangulations)
    D = np.zeros((m, m), dtype=float)

    for i in range(m):
        for j in range(i + 1, m):
            steps, _, _, _ = data.compute_pfd(i, j)  # (step, flip_list, i, j)
            D[i, j] = D[j, i] = float(steps)
    return D


# -------------------------------------------------------
# 2. 거리 행렬 D 위에서의 discrete 1-median
# -------------------------------------------------------

def discrete_1_median_from_D(D: np.ndarray,
                             candidates,
                             data_idx,
                             data_weights=None):
    """
    거리 행렬 D와 인덱스들만으로 discrete 1-median을 구한다.

    - candidates: center 후보 인덱스들의 리스트 (예: range(m))
    - data_idx: 비용을 계산할 데이터 인덱스들의 리스트
    - data_weights: 각 data_idx에 대응되는 가중치 (None이면 모두 1)

    반환:
      (best_idx, best_cost)
      best_idx  : center 후보 중 최적 인덱스
      best_cost : Σ_i w_i * D[best_idx, data_idx[i]]
    """
    candidates = np.asarray(candidates, dtype=int)
    data_idx = np.asarray(data_idx, dtype=int)

    if data_weights is None:
        w = np.ones(len(data_idx), dtype=float)
    else:
        w = np.asarray(data_weights, dtype=float)

    best_idx = -1
    best_cost = np.inf

    for c in candidates:
        dists = D[c, data_idx]           # shape = (|data_idx|,)
        cost = float((dists * w).sum())  # Σ w_i * d(c, data_i)
        if cost < best_cost:
            best_cost = cost
            best_idx = int(c)

    return best_idx, best_cost


# -------------------------------------------------------
# 3. (예시용) triangulation 코어셋 빌더 (아주 naive)
# -------------------------------------------------------

import numpy as np
import math

def build_triangulation_coreset_practical(
    data,
    eps: float = 0.1,
    alpha: float = 8.0,
    alpha_min: float = 16.0,
    center_candidates: int = 5,
):
    """
    triangulation 공간에서 compute_pfd를 거리로 사용하는
    1-median용 practical coreset 생성기.

    반환:
      S_idx      : 코어셋에 포함된 triangulation 인덱스들 (np.ndarray[int])
      S_weights  : 각 인덱스의 가중치 (np.ndarray[int])
    """

    n = len(data.triangulations)
    if n == 0:
        return np.array([], dtype=int), np.array([], dtype=int)

    # ---------------------------------------------------
    # 1. compute_pfd 거리 캐싱용 helper
    # ---------------------------------------------------
    pfd_cache = {}

    def get_pfd(i, j):
        if i == j:
            return 0.0
        a, b = (i, j) if i < j else (j, i)
        key = (a, b)
        if key not in pfd_cache:
            steps, _, _, _ = data.compute_pfd(a, b)
            pfd_cache[key] = float(steps)
        return pfd_cache[key]

    # ---------------------------------------------------
    # 2. "기하학적 중앙값" 대신 pfd 1-median 근사 center index 선택
    #    (처음 몇 개 triangulation 중에서 total pfd가 가장 작은 것)
    # ---------------------------------------------------
    m = n
    cand_num = min(m, center_candidates)
    best_center_idx = 0
    best_total_dist = float("inf")

    for c in range(cand_num):
        total = 0.0
        for j in range(m):
            total += get_pfd(c, j)
        if total < best_total_dist:
            best_total_dist = total
            best_center_idx = c

    center_idx = best_center_idx

    # ---------------------------------------------------
    # 3. center triangulation까지의 거리 배열 (nearest_dist) 계산
    #    (원래 코드의 ‖P[i] - x‖에 해당)
    # ---------------------------------------------------
    nearest_dist = np.zeros(m, dtype=float)
    for i in range(m):
        nearest_dist[i] = get_pfd(center_idx, i)

    nu_A = float(nearest_dist.sum())
    R = max(nu_A / max(1, m), 1e-12)
    Dmax = float(nearest_dist.max())

    # 모든 triangulation이 center와 너무 가깝다면 -> center 하나로 요약
    if Dmax < R:
        return np.array([center_idx], dtype=int), np.array([m], dtype=int)

    # ---------------------------------------------------
    # 4. 거리 스케일(ring)별로 버킷을 나누고, 각 버킷에서 대표 triangulation 1개 선택
    #    원래: v = P[i] - x, inf_norm = max(|v_k|)
    #    여기서는: v 대신 scalar = dist(center, i) 를 사용하여 1D grid로 단순화
    # ---------------------------------------------------
    from math import ceil, log2
    M = max(1, ceil(log2((Dmax + R) / R)))

    buckets = {}
    tiny = 1e-12

    for i in range(m):
        dist = nearest_dist[i]                  # center와의 거리 (pfd)
        inf_norm = dist                        # 1차원이라 그냥 dist 자체가 "norm"

        # level j 결정 (R, 2R, 4R, ... 거리대)
        if inf_norm < R:
            j = 0
        else:
            j = int(math.ceil(math.log2(max(inf_norm, tiny) / R)))
        j = min(j, M)

        # level j에서의 "cell 크기" rj
        rj = (eps * R * (2 ** j)) / alpha
        rj = max(rj, (eps * R) / alpha_min)

        # 1D grid cell index
        cell_1d = int(math.floor(inf_norm / rj))
        key = (j, cell_1d)

        # 같은 (j, cell_1d)에 속하는 triangulation들은 하나의 대표로 묶는다
        if key not in buckets:
            # 대표는 해당 버킷의 첫 triangulation i 로 두고, count = 1
            buckets[key] = [i, 1]
        else:
            buckets[key][1] += 1

    # ---------------------------------------------------
    # 5. 버킷에서 대표 인덱스, 가중치 추출
    # ---------------------------------------------------
    if buckets:
        rep_idx = []
        wts = []
        for rep, cnt in buckets.values():
            rep_idx.append(rep)
            wts.append(cnt)
        S_idx = np.array(rep_idx, dtype=int)
        S_weights = np.array(wts, dtype=int)
    else:
        # 이론상 거의 안 나오는 케이스지만 방어적으로
        S_idx = np.array([center_idx], dtype=int)
        S_weights = np.array([m], dtype=int)

    return S_idx, S_weights



# -------------------------------------------------------
# 4. 코어셋 기반 center vs 원본 기반 center를 비교
# -------------------------------------------------------

def compare_discrete_1median_with_coreset_pfd(D: np.ndarray,
                                              S_idx,
                                              S_weights):
    """
    거리 행렬 D와 (코어셋 인덱스, 가중치)을 받아서

    (1) 코어셋 기준 discrete 1-median:
        argmin_{c in P} Σ_{i in S} w_i * D[c, i]

    (2) 원본 기준 discrete 1-median:
        argmin_{c in P} Σ_{i in P} D[c, i]

    을 구한 뒤, 두 center를 모두 원본 P에서의 center_dist
    (Σ_{i in P} D[center, i])로 평가하여 비교한다.
    """
    m = D.shape[0]
    P_idx = np.arange(m, dtype=int)
    S_idx = np.asarray(S_idx, dtype=int)
    S_weights = np.asarray(S_weights, dtype=float)

    # (1) 코어셋 기준 center
    idx_c, _ = discrete_1_median_from_D(
        D, candidates=P_idx, data_idx=S_idx, data_weights=S_weights
    )

    # (2) 원본 기준 center
    idx_o, _ = discrete_1_median_from_D(
        D, candidates=P_idx, data_idx=P_idx, data_weights=None
    )

    # 두 center를 모두 "원본 전체 P"에서 평가
    cost_core_on_P = float(D[idx_c, P_idx].sum())
    cost_orig_on_P = float(D[idx_o, P_idx].sum())

    ratio = cost_core_on_P / cost_orig_on_P if cost_orig_on_P > 0 else np.nan

    result = {
        "center_coreset_idx": idx_c,
        "center_original_idx": idx_o,
        "cost_on_P_coreset": cost_core_on_P,
        "cost_on_P_original": cost_orig_on_P,
        "abs_diff": cost_core_on_P - cost_orig_on_P,
        "ratio": ratio,
        "rel_error": ratio - 1.0 if np.isfinite(ratio) else np.nan,
    }
    return result


# -------------------------------------------------------
# 5. 예시 실행 코드 (인스턴스: random_instance_4_40_2.json)
# -------------------------------------------------------

if __name__ == "__main__":
    # 인스턴스 경로는 실제 디렉토리 구조에 맞게 수정해줘
    # 예: "./data/benchmark_instances/random_instance_4_40_2.json"
    instance_path = "./data/benchmark_instances/random_instance_132_80_20.json"

    # data.py의 Data 클래스로 triangulation 로딩
    data = Data(instance_path)
    num_tris = len(data.triangulations)
    print(f"# of triangulations: {num_tris}")

    # (A) compute_pfd 기반 거리 행렬 D(i,j) 생성
    D = build_pfd_distance_matrix(data)

    # 코어셋 생성 (eps, alpha, alpha_min은 포인트 버전과 동일한 느낌으로)
    S_idx, S_weights = build_triangulation_coreset_practical(
        data,
        eps=0.1,
        alpha=8.0,
        alpha_min=16.0,
    )

    print(f"coreset size: {len(S_idx)}, indices: {S_idx}")
    print(f"coreset weights: {S_weights}")

    # (C) 코어셋 중심 vs 원본 중심의 center_dist 비교
    out = compare_discrete_1median_with_coreset_pfd(D, S_idx, S_weights)

    print("===== 결과 =====")
    print(f"원본 기준 1-median center index: {out['center_original_idx']}")
    print(f"코어셋 기준 1-median center index: {out['center_coreset_idx']}")
    print(f"원본 기준 최적 center_dist: {out['cost_on_P_original']:.6f}")
    print(f"코어셋 기반 center의 center_dist: {out['cost_on_P_coreset']:.6f}")
    print(f"절대 차이: {out['abs_diff']:.6f}")
    print(f"비율(coreset/original): {out['ratio']:.6f}")
    print(f"상대 오차(rel_error): {out['rel_error']:.6f}")


--------------------ReadData--------------------
num of pts: 80
num of triangulations: 20
Maximum Parallel flip distance: 9
Initial Center: 9 (total dist: 136)
# of triangulations: 20
0 -> 1 can be done in 7 step!
0 -> 2 can be done in 6 step!
0 -> 3 can be done in 6 step!
0 -> 4 can be done in 7 step!
0 -> 5 can be done in 4 step!
0 -> 6 can be done in 6 step!
0 -> 7 can be done in 7 step!
0 -> 8 can be done in 7 step!
0 -> 9 can be done in 8 step!
0 -> 10 can be done in 5 step!
0 -> 11 can be done in 5 step!
0 -> 12 can be done in 8 step!
0 -> 13 can be done in 6 step!
0 -> 14 can be done in 6 step!
0 -> 15 can be done in 7 step!
0 -> 16 can be done in 7 step!
0 -> 17 can be done in 6 step!
0 -> 18 can be done in 8 step!
0 -> 19 can be done in 7 step!
1 -> 2 can be done in 5 step!
1 -> 3 can be done in 5 step!
1 -> 4 can be done in 5 step!
1 -> 5 can be done in 5 step!
1 -> 6 can be done in 4 step!
1 -> 7 can be done in 5 step!
1 -> 8 can be done in 5 step!
1 -> 9 can be done in 5 st

In [3]:
import numpy as np

In [4]:
# ----- 비용 계산 유틸 -----
def cost_on_points(data_pts, center):
    """원본(무가중) 데이터에서의 비용 Σ‖x - c‖"""
    return float(np.linalg.norm(data_pts - center, axis=1).sum())

def weighted_cost_on_points(rep_pts, rep_w, center):
    """가중 데이터에서의 비용 Σ w_i‖s_i - c‖"""
    return float((rep_w * np.linalg.norm(rep_pts - center, axis=1)).sum())

In [5]:
# ----- discrete 1-median (후보 = C, 데이터 = D) -----
def discrete_1_median(candidates, data_pts, data_w=None, batch=0):
    """
    후보 집합 candidates 중에서, 데이터(data_pts, data_w)에 대한 합거리 비용을 최소화하는 후보를 반환.
    - data_w가 None이면 모두 가중치 1
    - 메모리 문제가 있으면 batch>0 으로 배치 계산 가능(후보를 배치로 나눠 처리)
    반환: (best_idx, best_center, best_cost_on_data)
    """
    C = candidates
    D = data_pts
    w = np.ones(len(D)) if data_w is None else data_w.astype(float)

    best_idx = -1
    best_cost = np.inf

    nC = len(C)
    if batch and batch < nC:
        # 배치 처리
        for start in range(0, nC, batch):
            end = min(start + batch, nC)
            Cb = C[start:end]                          # (B,d)
            # 거리: (B, |D|)
            dists = np.linalg.norm(Cb[:, None, :] - D[None, :, :], axis=2)
            costs = (dists * w[None, :]).sum(axis=1)   # (B,)
            j = np.argmin(costs)
            if costs[j] < best_cost:
                best_cost = float(costs[j])
                best_idx = start + int(j)
    else:
        # 전체 후보 한번에
        dists = np.linalg.norm(C[:, None, :] - D[None, :, :], axis=2)  # (|C|, |D|)
        costs = (dists * w[None, :]).sum(axis=1)                        # (|C|,)
        best_idx = int(np.argmin(costs))
        best_cost = float(costs[best_idx])

    return best_idx, C[best_idx], best_cost

In [6]:
# ----- 비교 래퍼 -----
def compare_discrete_1median_with_coreset(P, S_points, S_weights, batch=0):
    """
    (1) 코어셋 기준 discrete 1-median: argmin_{p∈P} Σ_i w_i‖p - s_i‖
    (2) 원본 기준 discrete 1-median: argmin_{p∈P} Σ_x∈P ‖p - x‖
    을 각각 구하고, 두 중심을 원본 P에서의 비용으로 비교한다.
    반환 딕셔너리:
      {
        'center_coreset': ...,      # 코어셋으로 구한 중심 (P 중 하나)
        'center_original': ...,     # 원본으로 구한 중심 (P 중 하나)
        'cost_on_P_coreset': ...,   # center_coreset의 원본 비용
        'cost_on_P_original': ...,  # center_original의 원본 비용(최적)
        'abs_diff': ...,            # 비용 차이
        'ratio': ...,               # cost_on_P_coreset / cost_on_P_original
        'rel_error': ...            # (ratio - 1)
      }
    """
    # (1) 코어셋으로 구한 discrete 1-median (후보=P, 데이터=S)
    idx_c, c_core, _ = discrete_1_median(
        candidates=P, data_pts=S_points, data_w=S_weights, batch=batch
    )

    # (2) 원본으로 구한 discrete 1-median (후보=P, 데이터=P)
    idx_o, c_orig, _ = discrete_1_median(
        candidates=P, data_pts=P, data_w=None, batch=batch
    )

    # 두 중심을 원본 데이터에서 평가
    cost_core_on_P = cost_on_points(P, c_core)
    cost_orig_on_P = cost_on_points(P, c_orig)

    ratio = cost_core_on_P / cost_orig_on_P if cost_orig_on_P > 0 else np.nan
    result = {
        'center_coreset': c_core,
        'center_original': c_orig,
        'cost_on_P_coreset': cost_core_on_P,
        'cost_on_P_original': cost_orig_on_P,
        'abs_diff': cost_core_on_P - cost_orig_on_P,
        'ratio': ratio,
        'rel_error': ratio - 1.0 if np.isfinite(ratio) else np.nan,
        'idx_coreset_in_P': idx_c,
        'idx_original_in_P': idx_o,
    }
    return result

In [7]:
# ----- 사용 예시 -----
if __name__ == "__main__":
    rng = np.random.default_rng(0)
    P = np.vstack([
        rng.normal(0, 1.0, size=(600, 2)),
        rng.normal(10, 1.2, size=(600, 2)),
    ])

    # 당신이 만든 코어셋 생성기를 사용하세요.
    # (아래는 이전 답변의 practical 버전 예시)
    def geometric_median_weizfeld(P, iters=30, eps=1e-8):
        x = P[np.random.randint(len(P))].astype(float)
        for _ in range(iters):
            diff = P - x
            dist = np.linalg.norm(diff, axis=1)
            w = 1.0 / np.maximum(dist, eps)
            x_new = (w[:, None] * P).sum(axis=0) / w.sum()
            if np.linalg.norm(x_new - x) < 1e-7:
                break
            x = x_new
        return x

    def build_1median_coreset_practical(P, eps=0.1, alpha=8.0, alpha_min=16.0):
        n, d = P.shape
        if n == 0:
            return P.copy(), np.array([], dtype=int)
        x = geometric_median_weizfeld(P)
        nearest_dist = np.linalg.norm(P - x, axis=1)
        nu_A = float(nearest_dist.sum())
        R = max(nu_A / max(1, n), 1e-12)
        Dmax = float(np.max(np.linalg.norm(P - x, axis=1)))
        if Dmax < R:
            return x[None, :], np.array([n], dtype=int)
        from math import ceil, log2
        M = max(1, ceil(log2((Dmax + R) / R)))
        buckets = {}
        tiny = 1e-12
        for i in range(n):
            v = P[i] - x
            inf_norm = np.max(np.abs(v))
            j = 0 if inf_norm < R else int(np.ceil(np.log2(max(inf_norm, tiny) / R)))
            j = min(j, M)
            rj = (eps * R * (2 ** j)) / alpha
            rj = max(rj, (eps * R) / alpha_min)
            cell_idx = tuple(np.floor(v / rj).astype(int))
            key = (j, cell_idx)
            if key not in buckets:
                buckets[key] = [P[i].copy(), 1]
            else:
                buckets[key][1] += 1
        reps, wts = zip(*buckets.values()) if buckets else ([], [])
        S_points = np.array(reps) if reps else x[None, :].copy()
        S_weights = np.array(wts, dtype=int) if wts else np.array([n], dtype=int)
        return S_points, S_weights

    # 코어셋 만들기
    S_points, S_weights = build_1median_coreset_practical(P, eps=0.4, alpha=8, alpha_min=16)

    # 비교
    out = compare_discrete_1median_with_coreset(P, S_points, S_weights, batch=0)
    print(f"# P 크기: {len(P)}, 코어셋 크기: {len(S_points)}, 가중치합: {S_weights.sum()}")
    print(f"원본 기준 최적 비용: {out['cost_on_P_original']:.6f}")
    print(f"코어셋으로 구한 중심의 원본 비용: {out['cost_on_P_coreset']:.6f}")
    print(f"절대 차이: {out['abs_diff']:.6f}")
    print(f"비율(coreset/original): {out['ratio']:.6f}, 상대오차: {out['rel_error']:.6f}")
    # 선택된 인덱스(둘 다 P의 한 점)
    print(f"원본-최적 중심 index: {out['idx_original_in_P']}, 코어셋-유도 중심 index: {out['idx_coreset_in_P']}")

# P 크기: 1200, 코어셋 크기: 318, 가중치합: 1200
원본 기준 최적 비용: 8640.212105
코어셋으로 구한 중심의 원본 비용: 8640.212105
절대 차이: 0.000000
비율(coreset/original): 1.000000, 상대오차: 0.000000
원본-최적 중심 index: 547, 코어셋-유도 중심 index: 547
